In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()


True

In [6]:
GoogleGeminiKey = os.getenv('GOOGLE_GEMINI_KEY')

In [7]:
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-001", api_key=GoogleGeminiKey)
llm.invoke("Write me a ballad about LangChain")

AIMessage(content="(Verse 1)\nIn realms of code, where dreams reside,\nA challenge loomed, a digital tide.\nTo link the worlds, of language deep,\nWith logic's flow, secrets to keep.\nThen LangChain rose, a guiding star,\nTo bridge the gap, however far.\n\n(Verse 2)\nWith Python's grace, and whispers keen,\nIt summoned forth the LLM machine.\nFrom OpenAI's halls, a word was heard,\nOf models vast, and knowledge stored.\nLangChain awoke, with purpose bright,\nTo harness power, and shed its light.\n\n(Verse 3)\nNo longer bound, by single quest,\nIt forged a chain, putting skills to the test.\nPrompt templates woven, with artful hand,\nTo guide the AI, across the land.\nFrom question asked, to answer found,\nLangChain's logic, knew no bound.\n\n(Verse 4)\nMemories it built, of chats long past,\nTo learn and grow, and knowledge cast.\nAgents it spawned, with tools to wield,\nTo search the web, and facts revealed.\nFrom databases deep, and spreadsheets wide,\nLangChain's agents, had nowhere

In [8]:
llm.invoke("The Sky is ?")


AIMessage(content="The sky is **blue** (most of the time, during the day).\n\nOf course, the sky can also be:\n\n*   **Black** (at night)\n*   **Gray** (when it's cloudy)\n*   **Orange, pink, or red** (at sunrise and sunset)\n*   **Purple** (sometimes at twilight)\n\nSo, the color of the sky depends on the time and weather!", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-001', 'safety_ratings': []}, id='run--32e40a4c-8bca-4549-9a33-829b54482d88-0', usage_metadata={'input_tokens': 4, 'output_tokens': 93, 'total_tokens': 97, 'input_token_details': {'cache_read': 0}})

##### Roles
- **User**: The user of the conversation.
- **Assistant**: The assistant of the conversation.
- **System**: The system that manages the conversation.
- **Chat**: 


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

prompt = [HumanMessage("What is the capital of France?")]

print(llm.invoke(prompt).content)


The capital of France is **Paris**.


In [12]:
system_msg = SystemMessage("""
You are a helpful assistant that help that respond to a question with exclamation marks.
""")

human_msg = HumanMessage("What is the capital of France?")
chat = [system_msg, human_msg]
llm.invoke(chat).content



'The capital of France is Paris!'

#### Templates

In [15]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("""
Answer the question on the based on the given context below. If the question cannot be answered using the information
provided in the context, respond with "I don't know".
Context: {context}
Question: {question}
Answer:
""")
prompt = template.invoke({
    "context": """ The measured temperature is at room temperature 20°C, the wifi STA and AP modes are turned on, and the battery is in the charging state. The test lasts for half an hour, and the highest temperature read through AXP2101 temperature data is 46 ℃. During the normal discharge process, the temperature will decrease by 3-4 ℃. If WiFi/Bluetooth function is not turned on, it can maintain a stable temperature of around 36 ℃
        This AMOLED screen can withstand high temperatures; at 40 to 60 degrees Celsius, it will not affect the screen display or touch function. In high temperature and high humidity conditions, there may be some polarization, which is within the normal range.
        For this product, it is recommended to use the low-power operation mode of ESP32-S3 to complete the application in some scenarios.""",
    "question": "How is the heat generation of this product, and will it affect the display?"

})
llm.invoke(prompt).content

'The product generates heat, reaching a maximum of 46°C during charging with WiFi STA and AP modes enabled for half an hour. During normal discharge, the temperature decreases by 3-4°C. Without WiFi/Bluetooth, it maintains around 36°C. The AMOLED screen can withstand temperatures between 40-60°C without affecting display or touch function, so the heat generation will not affect the display.'

#### ChatPrompt template

In [19]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", """
            Answer the question on the based on the given context below. If the question cannot be answered using the information
provided in the context, respond with "I don't know".
    """),
    ("human", "Context: {context}"),
    ("human", "Question: {question}")
])
template.invoke({
    "context": """
        This AMOLED screen can withstand high temperatures; at 40 to 60 degrees Celsius, it will not affect the screen display or touch function. In high temperature and high humidity conditions, there may be some polarization, which is within the normal range.
        For this product, it is recommended to use the low-power operation mode of ESP32-S3 to complete the application in some scenarios.""",
    "question": "How is the heat generation of this product, and will it affect the display?"

})
llm.invoke(prompt).content

"The product's heat generation can reach 46°C during charging with WiFi enabled. During normal discharge, the temperature decreases by 3-4°C. Without WiFi/Bluetooth, it maintains a stable temperature around 36°C. The AMOLED screen can withstand temperatures between 40 and 60 degrees Celsius without affecting display or touch function. Therefore, under the tested conditions, the heat generation will not affect the display."

#### Getting Specific Format out of LLMs

In [23]:
## Json
from langchain_core.pydantic_v1 import BaseModel

class AnswerWithJustification(BaseModel):
    """
        This class represents an answer with a justification.
    """
    answer: str
    justification: str

llm_v2 = ChatGoogleGenerativeAI(model="gemini-2.0-flash-001", api_key=GoogleGeminiKey,temperature=0)
structured_llm = llm_v2.with_structured_output(AnswerWithJustification)
(answer, justification) = structured_llm.invoke(prompt)
print(f"Answer: {answer}")
print(f"Justification: {justification}")

print("########### Example 2#############")
print(structured_llm.invoke("What weight more, a pound of brick or a pound of feathers?"))



Answer: ('answer', "The product's highest temperature read during charging and with WiFi on is 46°C. During normal discharge, the temperature decreases by 3-4°C. Without WiFi/Bluetooth, it stays around 36°C. The AMOLED screen can withstand temperatures between 40 to 60 degrees Celsius without affecting display or touch function.")
Justification: ('justification', "The context provides specific temperature readings under different conditions (charging, discharging, WiFi on/off) and states that the AMOLED screen is not affected by temperatures in the 40-60°C range. Therefore, the heat generation information is directly available, and the impact on the display can be assessed based on the screen's temperature tolerance.")
########### Example 2#############
answer='They weigh the same.' justification='A pound is a unit of weight, so a pound of anything weighs the same as a pound of anything else. The difference might be in volume or density, but not in weight'


In [24]:
# CSV
from langchain.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
parser.invoke("apple, banana, orange")



['apple', 'banana', 'orange']

#### Using Runnable Interface
The `Runnable` interface allows you to chain multiple LLMs together in a pipeline. This can be useful for creating complex workflows that involve multiple steps.

1. `invoke`: Single input, single output

2. `batch`: Multiple inputs, multiple outputs

3. `stream`: Single input, stream of outputs




In [30]:
#invoke method
print(llm.invoke("Hi there!").content)

#batch method
print(llm.batch(["Hello", "How are you?"]))


#stream method
for token in llm.stream("What is the capital of France?"):
    print(token.content)


Hi! How can I help you today?
[AIMessage(content='Hello! How can I help you today?', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-001', 'safety_ratings': []}, id='run--1426282c-cd90-4537-9e20-a1366fceb68d-0', usage_metadata={'input_tokens': 1, 'output_tokens': 10, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}}), AIMessage(content="I am doing well, thank you for asking! As a large language model, I don't experience emotions or feelings like humans do, but I am functioning optimally and ready to assist you. How can I help you today?", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash-001', 'safety_ratings': []}, id='run--d029e702-1b2a-4c2d-a1f1-e03d9884e39a-0', usage_metadata={'input_tokens': 4, 'output_tokens': 47, 'total_tokens': 51, 'input_token_det

#### Imperative Composition
its just a fancy name for chaining multiple LLMs together. It allows you to define a workflow that involves multiple steps and then execute it.


In [40]:
from langchain_core.runnables import  chain
from langchain_core.messages import ChatMessage


template = ChatPromptTemplate.from_messages([
    ("system","You are an AI assistant"),
    ("human", "{input}"),
])

@chain
def chatbot(values):
    prompt = template.invoke(values)
    response = llm.invoke(prompt)
    return response.content
print(chatbot.invoke({"input":"What is the capital of France?"}))




The capital of France is Paris.


In [41]:
@chain 
def chatbot(values):
    prompt = template.invoke(values)
    for token in llm.stream(prompt):
        yield token.content

for token in chatbot.stream({"input":"What is the capital of France?"}):
    print(token)




The
 capital of France
 is Paris.



### Declarative Composition
***LCEL***  declarative language for composing LangChain components. It allows you to define chains and flows using a simple syntax.


In [44]:
template = ChatPromptTemplate.from_messages([
    ("system","You are an AI assistant"),
    ("human", "{question}"),
])

chatbot = template|llm
chatbot.invoke({"question":"Which model provides offer LLMs"}).content


'There are several models that provide offer LLMs (Large Language Models), each with its own strengths and weaknesses. Here\'s a breakdown of some of the most prominent ones:\n\n**1. OpenAI Models:**\n\n*   **GPT-4:** This is currently OpenAI\'s flagship model and is known for its strong reasoning capabilities, creative text generation, and ability to handle complex tasks. It is considered one of the most powerful LLMs available. GPT-4 is available through the OpenAI API and powers services like ChatGPT Plus.\n*   **GPT-3.5:** A predecessor to GPT-4, GPT-3.5 is still a capable model and is often used when cost is a major factor. It\'s available through the OpenAI API and powers the free version of ChatGPT. There are several versions of GPT-3.5, with `gpt-3.5-turbo` being a popular and efficient choice.\n\n**2. Google Models:**\n\n*   **Gemini:** Google\'s most advanced model, designed to be multimodal (handling text, images, audio, and video). Gemini comes in different sizes:\n    *   